In [1]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box
import datetime
from pathlib import Path
import subprocess

In [2]:
# CELL 2 – Temporary workaround for r5py/Python 3.12 compatibility issue with Java/JPype (OPTIONAL IF NEEDED)
os.environ['JAVA_HOME'] = '/opt/homebrew/Cellar/openjdk@21/21.0.11'

# Patch jpype before importing to handle bytes return from getDefaultJVMPath
import jpype
original_getDefaultJVMPath = jpype.getDefaultJVMPath

def patched_getDefaultJVMPath(*args, **kwargs):
    result = original_getDefaultJVMPath(*args, **kwargs)
    if isinstance(result, bytes):
        result = result.decode('utf-8')
    return result

jpype.getDefaultJVMPath = patched_getDefaultJVMPath

# Import r5py after patching jpype
import r5py

In [3]:
# CELL 3 – Load data

OSM_FILE = "../../data/geo/Toronto.osm.pbf"
GTFS_FILES = [
    "../../data/geo/TTC Routes and Schedules Data.zip",
    "../../data/geo/GO-GTFS.zip",
    "../../data/geo/UP-GTFS.zip"
]
ORIGINS_FILE = "../../src/data/venues-centroids.geo.json"

In [ ]:
# CELL 4 - Set parameters

# Grid cell size (meters)
cell_size = 150

# Maximum commute time (minutes)
max_commute_time = 60

# Departure time
# Query different times and average, default to 7pm on a weeknight as default 
departure_time = datetime.datetime(
    2026, 6, 9, 
    14, 0, 0
    )

In [5]:
# CELL 5 – Create grid for commute time matrix calculation

# UTM 17N extent (meters)
minx = 604706
miny = 4819831
maxx = 657265
maxy = 4867556

# Create grid polygons
grid_cells = []

for x in np.arange(minx, maxx, cell_size):
    for y in np.arange(miny, maxy, cell_size):
        grid_cells.append(
            box(x, y, x + cell_size, y + cell_size)
        )

grid = gpd.GeoDataFrame(
    {"id": range(len(grid_cells))},
    geometry=grid_cells,
    crs="EPSG:26917"
)

# Remove cells that intersect with Lake Ontario (80%)
LAKE_ONTARIO = gpd.read_file("../../data/geo/lake-ontario.gpkg")
grid["intersection_area"] = grid.geometry.intersection(LAKE_ONTARIO.geometry[0]).area
grid = grid[grid["intersection_area"] / grid.geometry.area < 0.8].drop(columns="intersection_area")

grid = grid.to_crs("EPSG:4326")

# Create destinations GeoDataFrame with centroids
destinations = grid.copy()
destinations["geometry"] = destinations.geometry.centroid

/var/folders/69/g590bg750s935v88zgfrdm9w0000gn/T/ipykernel_18926/1515565343.py:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  destinations["geometry"] = destinations.geometry.centroid


In [6]:
# CELL 6 – Load origins
origins = gpd.read_file(ORIGINS_FILE)
origins["id"] = origins["id"].astype(str)
origins = origins.to_crs("EPSG:4326")

In [7]:
# CELL 7 – Build transport network
transport_network = r5py.TransportNetwork(
    OSM_FILE,
    GTFS_FILES
)
transport_network

In [8]:
# CELL 8 – Compute commute time matrix for each origin and merge as columns onto the grid
grid_time = grid.copy()

for _, origin in origins.iterrows():
    origin_id = origin["id"]
    origin_gdf = gpd.GeoDataFrame(
        [{"id": origin_id}],
        geometry=[origin.geometry],
        crs="EPSG:4326"
    )

    commute_time_matrix = r5py.TravelTimeMatrix(
        transport_network,
        origins=origin_gdf,
        destinations=destinations,
        departure=departure_time,
        transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
    )

    grid_time = grid_time.merge(
        commute_time_matrix[["to_id", "travel_time"]].rename(
            columns={"travel_time": f"commute_time_{origin_id}"}
        ),
        left_on="id",
        right_on="to_id",
        how="left"
    ).drop(columns="to_id")

    if max_commute_time is not None:
        grid_time[f"commute_time_{origin_id}"] = grid_time[f"commute_time_{origin_id}"].where(
            grid_time[f"commute_time_{origin_id}"] <= max_commute_time
        )

    print(f"Done: {origin_id}")

Done: 1
Done: 2
Done: 3
Done: 4
Done: 5
Done: 6
Done: 7
Done: 8
Done: 9
Done: 10
Done: 11
Done: 12
Done: 13
Done: 14
Done: 15
Done: 16
Done: 17
Done: 18
Done: 19
Done: 20
Done: 21
Done: 22
Done: 23
Done: 24
Done: 25
Done: 26
Done: 27
Done: 28
Done: 29
Done: 30
Done: 31
Done: 32
Done: 33
Done: 34
Done: 35
Done: 36
Done: 37
Done: 38
Done: 39
Done: 40
Done: 41
Done: 42
Done: 43
Done: 44
Done: 45
Done: 46
Done: 47
Done: 48
Done: 49
Done: 50
Done: 51
Done: 52
Done: 53
Done: 54
Done: 55
Done: 56
Done: 57
Done: 58
Done: 59
Done: 60
Done: 61
Done: 62
Done: 63
Done: 64
Done: 65
Done: 66
Done: 67
Done: 68
Done: 69
Done: 70
Done: 71
Done: 72
Done: 73
Done: 74
Done: 75
Done: 76
Done: 77
Done: 78
Done: 79
Done: 80
Done: 81
Done: 82
Done: 83


In [10]:
# CELL 9 – Drop cells with no commute time values for any origin
commute_cols = [col for col in grid_time.columns if col.startswith('commute_time_')]
grid_time_polygons = grid_time.dropna(subset=commute_cols, how='all').copy()

## Create point version using centroids (OPTIONAL)
# grid_time_points = grid_time_polygons.copy()
# grid_time_points['geometry'] = grid_time_points.geometry.centroid

grid_time_polygons.head()

,id,geometry,commute_time_1,commute_time_2,commute_time_3,commute_time_4,commute_time_5,commute_time_6,commute_time_7,commute_time_8,...,commute_time_74,commute_time_75,commute_time_76,commute_time_77,commute_time_78,commute_time_79,commute_time_80,commute_time_81,commute_time_82,commute_time_83
339,339,"POLYGON ((-79.70004 43.55104, -79.70001 43.552...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
340,340,"POLYGON ((-79.70001 43.55239, -79.69999 43.553...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
447,447,"POLYGON ((-79.6969 43.69687, -79.69687 43.6982...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,58.0,NaN,NaN,NaN
448,448,"POLYGON ((-79.69687 43.69822, -79.69684 43.699...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,58.0,NaN,NaN,NaN
659,659,"POLYGON ((-79.69816 43.55237, -79.69813 43.553...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# # CELL 10 – Bucket commute times into intervals

# # Time bucket interval (minutes), should divide into max_commute_time evenly
# interval = 15

# commute_cols = [c for c in grid_time.columns if c.startswith("commute_time_")]

# # Create bucket labels
# bins = np.arange(0, max_commute_time + interval, interval)
# labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins)-1)]

# for col in commute_cols:
#     grid_time[col] = pd.cut(
#         grid_time[col],
#         bins=bins,
#         labels=labels,
#         include_lowest=True,
#         right=True
#     )

# grid_time.head()

In [ ]:
# # CELL 11 – Dissolve grid cells by commute time buckets for each origin to create polygons representing areas of equal commute time
# dissolved_list = []

# for col in commute_cols:

#     temp = (
#         grid_time[[col, "geometry"]]
#         .dropna()
#         .dissolve(by=col)
#         .reset_index()
#     )

#     temp["commute_time_id"] = col
#     temp = temp.rename(columns={col: "bucket"})

#     dissolved_list.append(temp)

# dissolved = gpd.GeoDataFrame(
#     pd.concat(dissolved_list, ignore_index=True),
#     crs=grid_time.crs
# )

# dissolved['geometry'] = dissolved.geometry.set_precision(1e-5)

# dissolved.to_file(
#     "commute_time.geojson",
#     driver="GeoJSON"
# )

In [11]:
grid_time_polygons['geometry'] = grid_time_polygons.geometry.set_precision(1e-5)
grid_time_polygons.to_file('./commute_time.geojson', driver='GeoJSON')

In [ ]:
# # CELL 10 – Export grid as a point tileset

# OUTPUT_FILE = "../../data/geo/grid_time.geojson"
# PMTILES_FILE = "../../static/grid_time.pmtiles"

# subprocess.run([
#     "tippecanoe",
#     "-zg",
#     "-o", PMTILES_FILE,
#     "--no-feature-limit",
#     "--no-tile-size-limit",
#     OUTPUT_FILE,
#     "--force"
# ], check=True)

# Path(OUTPUT_FILE).unlink()

In [ ]:
# # CELL 9.2 – Export geometry and per-origin commute time CSVs

# OUTPUT_DIR = Path("../../data/geo/commute_times")
# OUTPUT_DIR.mkdir(exist_ok=True)

# # Geometry CSV (all grid cells, centroid coordinates)
# destinations["lon"] = destinations.geometry.x
# destinations["lat"] = destinations.geometry.y
# destinations[["id", "lon", "lat"]].round(5).to_csv(
#     "../../data/geo/grid_geometry.csv", index=False
# )

# # One CSV per commute time origin, nulls dropped
# commute_time_cols = [col for col in grid_time.columns if col.startswith("commute_time_")]

# for col in commute_time_cols:
#     grid_time[["id", col]].dropna().astype({"id": int, col: int}).to_csv(
#         OUTPUT_DIR / f"{col}.csv", index=False
#     )

# print(f"Exported geometry + {len(commute_time_cols)} commute time CSVs")

In [ ]:
# CELL 12 – Export grid as a tileset
INPUT_FILE = "./commute_time.geojson"

PMTILES_FILE = "../../static/commute_time.pmtiles"

subprocess.run([
    "tippecanoe",
    "-z17",
    "-o", PMTILES_FILE,
    "--extend-zooms-if-still-dropping",
    "--drop-densest-as-needed",
    "--no-tile-size-limit",
    INPUT_FILE,
    "--force"
], check=True)

Path(INPUT_FILE).unlink()

For layer 0, using name "commute_time"
32225 features, 10553263 bytes of geometry and attributes, 227178 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
  99.9%  17/36582/47770  


CompletedProcess(args=['tippecanoe', '-z17', '-o', '../../static/commute_time.pmtiles', '--extend-zooms-if-still-dropping', '--drop-densest-as-needed', '--no-tile-size-limit', './commute_time.geojson', '--force'], returncode=0)